# Determinants and Eigenspaces

- **Determinant** $\det A$: the factor by which $A$ scales area (volume in 3D). $\det A = 0 \iff A$ is **not invertible**.
- **Eigenvalue / eigenvector**: $A\mathbf v = \lambda\mathbf v$ with $\mathbf v \ne \mathbf 0$. $A$ only stretches $\mathbf v$.

In [ ]:
import sympy as sp
import numpy as np
from IPython.display import display

sp.init_printing()          # SymPy output rendered as math
lam = sp.symbols("lambda")

## Part A. Making matrices

Write a matrix as a **list of rows**.

In [ ]:
A = sp.Matrix([[1, 2],
               [3, 4]])                 # SymPy: exact
An = np.array([[1, 2],
               [3, 4]], dtype=float)    # NumPy: floats
display(A)
print(An)

**Special matrices:** identity (`eye`), diagonal (`diag`), zeros.

In [ ]:
display(sp.eye(3), sp.diag(1, 2, 3), sp.zeros(2, 3))
print(np.eye(3))
print(np.diag([1, 2, 3]))
print(np.zeros((2, 3)))

**Fill element by element.** Start from zeros and set entries with `A[i, j]` (row $i$, column $j$; indices start at $0$).

In [ ]:
G = sp.zeros(3, 3)
G[0, 1] = 5
G[2, 2] = sp.Rational(1, 3)     # exact 1/3
display(G)

Gn = np.zeros((3, 3))
Gn[0, 1] = 5
Gn[2, 2] = 1/3                  # float 0.333...
print(Gn)

**Fill from a formula** $a_{ij} = f(i, j)$. Example: $a_{ij} = i + j$.

In [ ]:
F = sp.Matrix(3, 3, lambda i, j: i + j)
display(F)

Fn = np.fromfunction(lambda i, j: i + j, (3, 3))
print(Fn)

### Symmetric and Hermitian matrices

- **Symmetric**: $A^T = A$ (mirror image across the diagonal).
- **Hermitian**: $A^* = A$, where $A^* = \overline{A}^{\,T}$ is the **conjugate transpose**. For real matrices, Hermitian = symmetric.

**Why it matters:** symmetric/Hermitian matrices always have **real eigenvalues** and a full set of eigenvectors.

| | SymPy | NumPy |
|---|---|---|
| transpose | `A.T` | `A.T` |
| conjugate transpose | `A.H` | `A.conj().T` |
| symmetric? | `A.is_symmetric()` | `np.allclose(A, A.T)` |
| Hermitian? | `A.is_hermitian` | `np.allclose(A, A.conj().T)` |

In SymPy, $i$ is `sp.I`; in NumPy it is `1j`. Note `is_hermitian` has **no** parentheses.

In [ ]:
Sym = sp.Matrix([[2, 1],
                 [1, 3]])
H = sp.Matrix([[2,        1 - sp.I],
               [1 + sp.I, 3       ]])
display(Sym, H)

print("Sym symmetric?", Sym.is_symmetric(), "  Hermitian?", Sym.is_hermitian)
print("H   symmetric?", H.is_symmetric(),   "  Hermitian?", H.is_hermitian)

Hn = np.array([[2, 1 - 1j],
               [1 + 1j, 3]])
print("NumPy: H Hermitian?", np.allclose(Hn, Hn.conj().T))

The eigenvalues of $H$ are real, even though $H$ has complex entries:

In [ ]:
H.eigenvals()

## Part B. Determinant

For a $2\times 2$ matrix:
$$ \det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc. $$

**Example.** $\det\begin{pmatrix} 3 & 1 \\ 2 & 4 \end{pmatrix} = 3\cdot 4 - 1\cdot 2 = 10$. In SymPy: `A.det()`.

In [ ]:
A = sp.Matrix([[3, 1],
               [2, 4]])
A.det()

SymPy also works with **symbols**, so it can derive the formula itself:

In [ ]:
a, b, c, d = sp.symbols("a b c d")
Ab = sp.Matrix([[a, b],
                [c, d]])
display(Ab)
Ab.det()

**3×3 example.** Same command for any size:

In [ ]:
B = sp.Matrix([[2, 0, 1],
               [1, 3, 2],
               [1, 1, 2]])
display(B)
B.det()

**A singular matrix.** Consider
$$ S = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 5 & 7 & 9 \end{pmatrix}. $$
Row 3 is the sum of rows 1 and 2, so the rows are dependent and $\det S = 0$.

In [ ]:
S = sp.Matrix([[1, 2, 3],
               [4, 5, 6],
               [5, 7, 9]])
S.det()

**NumPy pitfall:** the same determinant in floating point is not exactly $0$. Never test `det == 0`; test `abs(det) < 1e-10`.

In [ ]:
np.linalg.det(np.array(S, dtype=float))

## Part C. Eigenvalues and eigenspaces

$A\mathbf v = \lambda\mathbf v$ means $(A - \lambda I)\mathbf v = \mathbf 0$. A nonzero $\mathbf v$ exists exactly when
$$ \det(A - \lambda I) = 0 \qquad \text{(characteristic equation)}. $$

The **eigenspace** of $\lambda$ is $E_\lambda = \operatorname{Null}(A - \lambda I)$: all eigenvectors for $\lambda$, plus $\mathbf 0$.

**Example.** $A = \begin{pmatrix} 4 & 1 \\ 2 & 3 \end{pmatrix}$.

**Characteristic polynomial, two ways:**
- `(A - lam*Id).det()` (with `Id = sp.eye(n)`) is $\det(A - \lambda I)$, as in the textbook.
- `A.charpoly(lam)` is SymPy's built-in. It computes $\det(\lambda I - A)$, which differs by a factor $(-1)^n$ ($n\times n$ matrix). **Same roots, same eigenvalues.** Use `.as_expr()` to get a normal expression.

In [ ]:
A = sp.Matrix([[4, 1],
               [2, 3]])
Id = sp.eye(2)          # 'Id', not 'I': sp.I is the imaginary unit

p1 = (A - lam*Id).det()
p2 = A.charpoly(lam).as_expr()
display(sp.factor(p1), sp.factor(p2))

For the $3\times 3$ matrix $B$ from Part B ($n$ odd), the two differ by a sign:

In [ ]:
display(sp.factor((B - lam*sp.eye(3)).det()), sp.factor(B.charpoly(lam).as_expr()))

**Eigenvalues:** roots of the characteristic polynomial. **Eigenspaces:** null space of $A - \lambda I$.

In [ ]:
eigs = sp.solve(p1, lam)
display(eigs)
for l in eigs:
    print(f"lambda = {l}: basis of eigenspace")
    display((A - l*Id).nullspace())

**Shortcuts:**
- `A.eigenvals()` $\to$ `{eigenvalue: algebraic multiplicity}`
- `A.eigenvects()` $\to$ list of `(eigenvalue, algebraic multiplicity, [basis of eigenspace])`

In [ ]:
display(A.eigenvals())
display(A.eigenvects())

### Algebraic vs geometric multiplicity

- **Algebraic**: how many times $\lambda$ is a root of the characteristic polynomial.
- **Geometric**: $\dim E_\lambda$ (number of basis vectors).

Always geometric $\le$ algebraic. Both matrices below have $\lambda = 2$ twice:

In [ ]:
D1 = sp.Matrix([[2, 0],
                [0, 2]])
D2 = sp.Matrix([[2, 1],
                [0, 2]])

for name, D in [("D1", D1), ("D2", D2)]:
    for l, alg, basis in D.eigenvects():
        print(f"{name}: lambda={l}, algebraic={alg}, geometric={len(basis)}")

**Why they differ:**
- $D_1 = 2\,\mathrm{Id}$, so $D_1\mathbf v = 2\mathbf v$ for **every** $\mathbf v$. The eigenspace $E_2$ is all of $\mathbb R^2$, with basis $(1,0), (0,1)$: dimension $2$.
- For $D_2$: $(D_2 - 2\,\mathrm{Id})\mathbf v = (v_2, 0) = \mathbf 0$ forces $v_2 = 0$. So $E_2$ is spanned by $(1, 0)$ only: dimension $1$.

So $D_2$ has only **one** independent eigenvector, not enough for a basis of $\mathbb R^2$. Such a matrix cannot be diagonalized. You can always find the geometric multiplicity by counting the vectors in `nullspace()`:

In [ ]:
display((D1 - 2*sp.eye(2)).nullspace(), (D2 - 2*sp.eye(2)).nullspace())

### NumPy: `np.linalg.eig`

`vals, vecs = np.linalg.eig(A)`

**Rule: eigenvectors are the COLUMNS of `vecs`.** The eigenvector for `vals[i]` is `vecs[:, i]`, **not** `vecs[i]`. Taking rows gives wrong vectors with no error.

NumPy scales eigenvectors to length $1$, so they may differ from SymPy's by a constant factor. For symmetric/Hermitian matrices, `np.linalg.eigh` is better: real eigenvalues, sorted.

In [ ]:
An = np.array([[4, 1],
               [2, 3]], dtype=float)
vals, vecs = np.linalg.eig(An)
vals, vecs = vals.real, vecs.real       # NumPy may return complex type with 0 imaginary part

for i in range(len(vals)):
    v = vecs[:, i]                      # column i
    print(f"lambda={vals[i]:.1f}, v={v.round(4)},  Av - lambda v = {(An @ v - vals[i]*v).round(12)}")

## Exercises

Solve with **SymPy** (exact); check with **NumPy** where asked.

**Syntax you will need:**

| task | SymPy | NumPy |
|---|---|---|
| make a matrix | `sp.Matrix([[1, 2], [3, 4]])` | `np.array([[1, 2], [3, 4]], dtype=float)` |
| identity / diagonal | `sp.eye(n)`, `sp.diag(a, b, c)` | `np.eye(n)`, `np.diag([a, b, c])` |
| symbol | `k = sp.symbols("k")` | |
| show nicely | `display(expr)` | `print(A)` |
| determinant | `A.det()` | `np.linalg.det(A)` |
| solve `expr` $= 0$ | `sp.solve(expr, k)` | |
| char. polynomial | `(A - lam*sp.eye(n)).det()` or `A.charpoly(lam).as_expr()` | |
| factor | `sp.factor(expr)` | |
| eigenvalues | `A.eigenvals()` | `np.linalg.eigvals(A)` |
| eigenvectors | `A.eigenvects()` or `(A - l*sp.eye(n)).nullspace()` | `vals, vecs = np.linalg.eig(A)`; column `vecs[:, i]` |
| symmetric? | `A.is_symmetric()` | `np.allclose(A, A.T)` |
| Hermitian? | `A.is_hermitian` | `np.allclose(A, A.conj().T)` |

**Exercise 1.** Let
$$ M = \begin{pmatrix} 2 & 1 & 1 \\ 1 & 2 & 1 \\ 1 & 1 & 2 \end{pmatrix}. $$

Use **SymPy** for (a)–(d).

(a) Find the **determinant** of $M$.

(b) Find the **eigenvalues** of $M$ and their algebraic multiplicities.

(c) Find the **eigenvectors**: a basis of each eigenspace. What is the geometric multiplicity of each eigenvalue?

(d) Is $M$ symmetric? Are your eigenvalues real, as expected?

(e) Reproduce (a)–(d) with **NumPy**.

In [ ]:
# Exercise 1

**Exercise 2.** Let
$$ A = \begin{pmatrix} k & 1 & 0 \\ 1 & k & 0 \\ 0 & 0 & 2 \end{pmatrix}. $$

Use **SymPy**.

(a) Compute $\det A$ as a function of $k$.

(b) Find the characteristic polynomial and **factor** it. Then **solve** it to get the eigenvalues as functions of $k$. For which $k$ is one of the eigenvalues $0$? Those are the $k$ where $A$ is singular: check that they match the zeros of $\det A$ from (a).

In [ ]:
# Exercise 2